In [1]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt #for igures
%matplotlib inline

In [2]:
# read in all the words
words = open('names.txt', 'r').read().splitlines()
words[:8] #check

['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia']

In [3]:
#built wocabulary of characters and mappings to from integers 
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i, s in enumerate(chars)}
stoi['.']= 0
itos={i:s for s, i in stoi.items()}
print(itos)

{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}


In [4]:
# buil the dataset
block_size=3 #context lenght :how many charakter do we take to predict next one 
X , Y = [] , []
for w in words[:5]:#firstly try with only 5 word = 32 sample
    print(w)
    contex = [0] * block_size
    for ch in w + '.' :
        ix =stoi[ch]
        X.append(contex)
        Y.append(ix)
        print(''.join(itos[i] for i in contex), '---->',itos[ix])
        contex=contex[1:]+[ix] #crop and append

X = torch.tensor(X)
Y = torch.tensor(Y)

emma
... ----> e
..e ----> m
.em ----> m
emm ----> a
mma ----> .
olivia
... ----> o
..o ----> l
.ol ----> i
oli ----> v
liv ----> i
ivi ----> a
via ----> .
ava
... ----> a
..a ----> v
.av ----> a
ava ----> .
isabella
... ----> i
..i ----> s
.is ----> a
isa ----> b
sab ----> e
abe ----> l
bel ----> l
ell ----> a
lla ----> .
sophia
... ----> s
..s ----> o
.so ----> p
sop ----> h
oph ----> i
phi ----> a
hia ----> .


In [5]:
X.shape,X.dtype, Y.shape,Y.dtype

(torch.Size([32, 3]), torch.int64, torch.Size([32]), torch.int64)

In [6]:
#look up table 
#in the paper mention on the video , in 25000 word each word haves their 30 dimension embeddings
#for our neural network we have 3 charecter to predict nex one from 27 probable character
#so we should do embed them into small number of dimension like 2 
#as well as our network train the characters probabilities will move in the 2 dimention space and get far and naer with each other
#lets look up table for each 27 charecter for deir embeddings

C= torch.randn(27,2)

In [7]:
C[5]

tensor([-1.5269, -1.3261])

In [8]:
# We can represent each sample using F.one_hot.
# one_hot produces a vector of length 27 for each index:
# every position is 0, except the position at that index, which is 1.
F.one_hot(torch.tensor(5),num_classes=27  )


tensor([0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0])

In [9]:
#we had to casting float to multipy a float with int
#matrix dimensions will be [1,27][27,2] = [1,2]
F.one_hot(torch.tensor(5),num_classes=27  ).float() @ C

tensor([-1.5269, -1.3261])

In [10]:
#we will contunie with just index them
#C[[5,6,7]]
C[torch.tensor([5,6,7,7,6,6,6,5])]
#C[X].shape


tensor([[-1.5269, -1.3261],
        [-0.4952, -0.7533],
        [-0.5269,  0.7048],
        [-0.5269,  0.7048],
        [-0.4952, -0.7533],
        [-0.4952, -0.7533],
        [-0.4952, -0.7533],
        [-1.5269, -1.3261]])

In [11]:
# every character index in X is replaced by its 2-dimensional vector (row) from C
emb=C[X] #C[X][2] created = C [32,3,2] 
emb.shape

torch.Size([32, 3, 2])

In [12]:
#lets add a hidden layer
w1= torch.randn((6,100)) #100neuron is optional
b1=torch.randn(100)

In [13]:
# emb @ w1 + b1
emb[:, 0, :]##each 32 target samples first embed in 3 previous



tensor([[ 0.0747, -0.7490],
        [ 0.0747, -0.7490],
        [ 0.0747, -0.7490],
        [-1.5269, -1.3261],
        [-0.0593,  1.2396],
        [ 0.0747, -0.7490],
        [ 0.0747, -0.7490],
        [ 0.0747, -0.7490],
        [ 0.6773,  0.9783],
        [ 1.5249, -1.3094],
        [ 0.7189,  0.2051],
        [-0.8873,  0.3194],
        [ 0.0747, -0.7490],
        [ 0.0747, -0.7490],
        [ 0.0747, -0.7490],
        [-0.0459, -0.8620],
        [ 0.0747, -0.7490],
        [ 0.0747, -0.7490],
        [ 0.0747, -0.7490],
        [ 0.7189,  0.2051],
        [-0.1528, -0.3252],
        [-0.0459, -0.8620],
        [-0.0744,  1.6195],
        [-1.5269, -1.3261],
        [ 1.5249, -1.3094],
        [ 0.0747, -0.7490],
        [ 0.0747, -0.7490],
        [ 0.0747, -0.7490],
        [-0.1528, -0.3252],
        [ 0.6773,  0.9783],
        [-0.5775, -0.0400],
        [ 0.4787, -1.0174]])

In [14]:
torch.cat([emb[:,0,:], emb[:,1,:],   emb[:,2,:]], 1).shape 
#each row represent one targets  [x1,x2,x3,y1,y2,y3] embeds, x and y is dimentions 123 is previous points embeds


torch.Size([32, 6])

In [15]:

torch.cat(torch.unbind(emb,1),1)
#each row represent one targets  [x1,x2,x3,y1,y2,y3] embeds, x and y is dimentions 123 is previous points embeds
#this way creates a hole new memory so this way is not efficient

tensor([[ 0.0747, -0.7490,  0.0747, -0.7490,  0.0747, -0.7490],
        [ 0.0747, -0.7490,  0.0747, -0.7490, -1.5269, -1.3261],
        [ 0.0747, -0.7490, -1.5269, -1.3261, -0.0593,  1.2396],
        [-1.5269, -1.3261, -0.0593,  1.2396, -0.0593,  1.2396],
        [-0.0593,  1.2396, -0.0593,  1.2396, -0.0459, -0.8620],
        [ 0.0747, -0.7490,  0.0747, -0.7490,  0.0747, -0.7490],
        [ 0.0747, -0.7490,  0.0747, -0.7490,  0.6773,  0.9783],
        [ 0.0747, -0.7490,  0.6773,  0.9783,  1.5249, -1.3094],
        [ 0.6773,  0.9783,  1.5249, -1.3094,  0.7189,  0.2051],
        [ 1.5249, -1.3094,  0.7189,  0.2051, -0.8873,  0.3194],
        [ 0.7189,  0.2051, -0.8873,  0.3194,  0.7189,  0.2051],
        [-0.8873,  0.3194,  0.7189,  0.2051, -0.0459, -0.8620],
        [ 0.0747, -0.7490,  0.0747, -0.7490,  0.0747, -0.7490],
        [ 0.0747, -0.7490,  0.0747, -0.7490, -0.0459, -0.8620],
        [ 0.0747, -0.7490, -0.0459, -0.8620, -0.8873,  0.3194],
        [-0.0459, -0.8620, -0.8873,  0.3

In [16]:
emb.shape[0]# number of data samples

32

In [17]:
emb.view(emb.shape[0],6)#this is more efficient to pytorch tensor structure 


#emb.view(-1,6) will also work   # ? × 6 = 192 -->32
#.view extremly efficient

tensor([[ 0.0747, -0.7490,  0.0747, -0.7490,  0.0747, -0.7490],
        [ 0.0747, -0.7490,  0.0747, -0.7490, -1.5269, -1.3261],
        [ 0.0747, -0.7490, -1.5269, -1.3261, -0.0593,  1.2396],
        [-1.5269, -1.3261, -0.0593,  1.2396, -0.0593,  1.2396],
        [-0.0593,  1.2396, -0.0593,  1.2396, -0.0459, -0.8620],
        [ 0.0747, -0.7490,  0.0747, -0.7490,  0.0747, -0.7490],
        [ 0.0747, -0.7490,  0.0747, -0.7490,  0.6773,  0.9783],
        [ 0.0747, -0.7490,  0.6773,  0.9783,  1.5249, -1.3094],
        [ 0.6773,  0.9783,  1.5249, -1.3094,  0.7189,  0.2051],
        [ 1.5249, -1.3094,  0.7189,  0.2051, -0.8873,  0.3194],
        [ 0.7189,  0.2051, -0.8873,  0.3194,  0.7189,  0.2051],
        [-0.8873,  0.3194,  0.7189,  0.2051, -0.0459, -0.8620],
        [ 0.0747, -0.7490,  0.0747, -0.7490,  0.0747, -0.7490],
        [ 0.0747, -0.7490,  0.0747, -0.7490, -0.0459, -0.8620],
        [ 0.0747, -0.7490, -0.0459, -0.8620, -0.8873,  0.3194],
        [-0.0459, -0.8620, -0.8873,  0.3

In [18]:
emb.view(32, 6) == torch.cat(torch.unbind(emb, 1), 1)
#in this way we get the exact same resul you can check with run above code line


tensor([[True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, T

In [19]:
h = torch.tanh(emb.view(-1,6) @ w1 +b1) #this is hidden layer with ativation
h.shape #32 train data -> 32 hidden layer consist of 100 neurons

torch.Size([32, 100])

In [20]:
w2= torch.randn((100,27))
b2= torch.randn(27)

In [21]:
logits= h @ w2 + b2
logits.shape# final layer

torch.Size([32, 27])

In [22]:
counts= logits.exp() #for negatives 

In [23]:
prob = counts / counts.sum(1, keepdims= True)# for each input sample there is 27 charecters probabilities for next come 

prob.shape
#now thats gives us the a row of 27 probibilites for each sample.
#max prob will be our max likely prediction

torch.Size([32, 27])

In [24]:
prob[0].sum()

tensor(1.0000)

In [25]:
torch.arange(32)

tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
        18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31])

In [26]:
#this cell work when we select first 32 sample
prob[torch.arange(32), Y] #only correct chars probability for each sample
# each row and Y in index of probability of correct characters

loss = -prob[torch.arange(32), Y].log().mean()
#we need log and mean to find loss
loss
#al these are not efficient we will use cross enthrophy but main idea is same

tensor(12.9278)

In [27]:
# for exact same result we can :
loss =F.cross_entropy(logits, Y)
loss



tensor(12.9278)

In [28]:
g = torch.Generator().manual_seed(2147483647) # for reproducibility
C = torch.randn((27, 2), generator=g)
w1 = torch.randn((6, 100), generator=g)
b1 = torch.randn(100, generator=g)
w2 = torch.randn((100, 27), generator=g)
b2 = torch.randn(27, generator=g)
parameters = [C, w1, b1, w2, b2]

In [35]:
sum(p.nelement() for p in parameters)#number of all parameters in the network

3481

In [36]:
parameters = [C, w1, b1, w2, b2]
for p in parameters:
    p.requires_grad = True

In [31]:

for _ in range (100):
    #Forwardpass
    emb=C[X] #[32,3,2]
    h=torch.tanh(emb.view(-1,6)@ w1 + b1)
    logits = h @ w2 + b2
    loss = F.cross_entropy(logits, Y)
    print (loss.item())
    #backward pass
    for p in parameters:
        p.grad = None
    loss.backward()
    #update
    for p in parameters:
        p.data += -0.1 *p.grad


17.76971435546875
13.656400680541992
11.298768997192383
9.452457427978516
7.984262466430664
6.891322135925293
6.100014686584473
5.452037334442139
4.8981523513793945
4.414664268493652
3.985849142074585
3.6028313636779785
3.2621419429779053
2.961380958557129
2.6982979774475098
2.469712972640991
2.271660566329956
2.101283550262451
1.9571768045425415
1.8374860286712646
1.7380967140197754
1.6535111665725708
1.579089879989624
1.5117666721343994
1.449605107307434
1.3913122415542603
1.335992455482483
1.283052921295166
1.2321909666061401
1.1833813190460205
1.1367987394332886
1.092664361000061
1.051092505455017
1.0120269060134888
0.9752705097198486
0.9405566453933716
0.9076125621795654
0.8761922717094421
0.8460891246795654
0.8171359896659851
0.78919917345047
0.762174665927887
0.7359815239906311
0.7105580568313599
0.6858612298965454
0.6618652939796448
0.6385657787322998
0.6159818172454834
0.5941659212112427
0.5732107162475586
0.5532563924789429
0.5344882011413574
0.5171169638633728
0.501331448554

In [38]:
logits.max(1)#for every dimention 

torch.return_types.max(
values=tensor([11.3379, 13.0894, 18.9434, 17.5786, 12.7205, 11.3379, 13.0234, 11.5993,
        13.5442, 15.5540, 12.5548, 17.6663, 11.3379, 12.6919, 13.9530, 17.0410,
        11.3379, 13.8052, 11.4075, 13.1371, 15.8441, 12.1391,  7.9167,  7.8798,
        13.8772, 11.3379, 13.0983, 13.6820, 11.2885, 14.2338, 15.6165, 12.0455],
       grad_fn=<MaxBackward0>),
indices=tensor([ 1, 13, 13,  1,  0,  1, 12,  9, 22,  9,  1,  0,  1, 22,  1,  0,  1, 19,
         1,  2,  5, 12, 12,  1,  0,  1, 15, 16,  8,  9,  1,  0]))

In [39]:
Y

tensor([ 5, 13, 13,  1,  0, 15, 12,  9, 22,  9,  1,  0,  1, 22,  1,  0,  9, 19,
         1,  2,  5, 12, 12,  1,  0, 19, 15, 16,  8,  9,  1,  0])

In [34]:
torch.randint(0, X.shape[0], (32,))

tensor([23,  6, 24, 25, 13, 17, 14, 19, 10, 19,  9, 11, 27,  4, 14, 31, 29,  3,
         3,  8, 26, 12, 15, 12, 17,  1, 12,  3, 31, 12,  9, 13])